# Bangkok Airbnb EDA
## What Makes a Successful Airbnb Listing in Bangkok?

**Dataset:** Inside Airbnb — Bangkok, Central Thailand (September 2025)  
**Files:** listings.csv.gz, calendar.csv.gz, reviews.csv.gz, neighbourhoods.geojson

### Guiding Questions
1. What drives listing price?
2. Which neighborhoods are most competitive?
3. What makes a high-performing host?
4. How do room type and seasonality affect price and availability?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore') # suppress pandas/geopandas deprecation warnings

# Plot settings
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

listings = pd.read_csv('../data/listings.csv.gz', compression='gzip')
calendar = pd.read_csv('../data/calendar.csv.gz', compression='gzip')
reviews = pd.read_csv('../data/reviews.csv.gz', compression='gzip')
neighbourhoods = gpd.read_file('../data/neighbourhoods.geojson')

print(f"Listings:      {listings.shape}")
print(f"Calendar:      {calendar.shape}")
print(f"Reviews:       {reviews.shape}")
print(f"Neighbourhoods:{neighbourhoods.shape}")

Listings:      (28806, 79)
Calendar:      (10514202, 7)
Reviews:       (583333, 6)
Neighbourhoods:(50, 3)


In [2]:
print(f"Shape: {listings.shape}")
print(f"\nColumn names:\n {listings.columns.tolist()}")

Shape: (28806, 79)

Column names:
 ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability

In [3]:
# Categorize columns by type/purpose
col_groups = {
    'identifiers':    ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'picture_url'],
    'listing_info':   ['name', 'description', 'neighborhood_overview', 'property_type', 'room_type',
                       'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities'],
    'host_info':      ['host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
                       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
                       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
                       'host_neighbourhood', 'host_listings_count', 'host_total_listings_count',
                       'host_verifications', 'host_has_profile_pic', 'host_identity_verified'],
    'location':       ['neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
                       'latitude', 'longitude'],
    'pricing':        ['price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
                       'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights',
                       'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm'],
    'availability':   ['has_availability', 'availability_30', 'availability_60', 'availability_90',
                       'availability_365', 'availability_eoy', 'calendar_updated', 'calendar_last_scraped'],
    'reviews':        ['number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
                       'number_of_reviews_ly', 'first_review', 'last_review', 'review_scores_rating',
                       'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
                       'review_scores_communication', 'review_scores_location', 'review_scores_value',
                       'reviews_per_month'],
    'business':       ['license', 'instant_bookable', 'calculated_host_listings_count',
                       'calculated_host_listings_count_entire_homes',
                       'calculated_host_listings_count_private_rooms',
                       'calculated_host_listings_count_shared_rooms',
                       'estimated_occupancy_l365d', 'estimated_revenue_l365d']
}

for group, cols in col_groups.items():
    print(f"{group.upper():20} ({len(cols)} cols): {cols[:4]}{'...' if len(cols) > 4 else ''}")

IDENTIFIERS          (6 cols): ['id', 'listing_url', 'scrape_id', 'last_scraped']...
LISTING_INFO         (11 cols): ['name', 'description', 'neighborhood_overview', 'property_type']...
HOST_INFO            (18 cols): ['host_id', 'host_url', 'host_name', 'host_since']...
LOCATION             (5 cols): ['neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude']...
PRICING              (9 cols): ['price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights']...
AVAILABILITY         (8 cols): ['has_availability', 'availability_30', 'availability_60', 'availability_90']...
REVIEWS              (14 cols): ['number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'number_of_reviews_ly']...
BUSINESS             (8 cols): ['license', 'instant_bookable', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes']...


In [4]:
print("DATA TYPES")
print(listings.dtypes.value_counts())
print(f"\nTotal columns: {listings.shape[1]}")

DATA TYPES
str        34
float64    25
int64      20
Name: count, dtype: int64

Total columns: 79


In [6]:
missing=listings.isnull().sum()
missing_pct=(missing / len(listings) * 100).round(1)
missing_df=pd.DataFrame({'missing_count': missing, 
                         'missing_pct': missing_pct
                         }).query('missing_count>0').sort_values('missing_pct', ascending=False)
print(f"Columns WITH missing values: {len(missing_df)} out of {listings.shape[1]}")
print(f"Columns with NO missing values: {listings.shape[1] - len(missing_df)}\n")
print(missing_df.to_string())


Columns WITH missing values: 43 out of 79
Columns with NO missing values: 36

                              missing_count  missing_pct
license                               28806        100.0
calendar_updated                      28806        100.0
neighbourhood_group_cleansed          28806        100.0
host_neighbourhood                    19436         67.5
neighborhood_overview                 19322         67.1
neighbourhood                         19322         67.1
host_about                            11990         41.6
review_scores_cleanliness             10091         35.0
reviews_per_month                     10090         35.0
review_scores_value                   10096         35.0
first_review                          10090         35.0
review_scores_rating                  10090         35.0
review_scores_accuracy                10090         35.0
review_scores_checkin                 10093         35.0
review_scores_communication           10091         35.0
review_sco

In [7]:
cols_to_drop = [
    'license',                      # 100% missing
    'calendar_updated',             # 100% missing
    'neighbourhood_group_cleansed', # 100% missing
    'neighbourhood',                # 67% missing — neighbourhood_cleansed is better
    'listing_url',                  # not needed for analysis
    'scrape_id',                    # metadata
    'picture_url',                  # metadata
    'host_url',                     # metadata
    'host_thumbnail_url',           # metadata
    'host_picture_url',             # metadata
]

listings = listings.drop(columns=cols_to_drop)
print(f"Columns after dropping: {listings.shape[1]}")
print(f"Dropped: {len(cols_to_drop)} columns")

Columns after dropping: 69
Dropped: 10 columns


In [8]:
# Check what price looks like before cleaning
print(listings['price'].dtype)
print(listings['price'].head(10))

str
0    $1,595.00
1          NaN
2          NaN
3    $4,188.00
4    $1,450.00
5    $1,368.00
6          NaN
7    $5,600.00
8    $1,147.00
9    $1,416.00
Name: price, dtype: str


In [9]:
# Clean price column
listings['price'] = (listings['price']
                     .str.replace('$', '', regex=False)
                     .str.replace(',', '', regex=False)
                     .astype(float))

# Check result
print(f"dtype after cleaning: {listings['price'].dtype}")
print(f"Missing prices: {listings['price'].isnull().sum()}")
print(f"\nPrice statistics:")
print(listings['price'].describe().round(2))

dtype after cleaning: float64
Missing prices: 5533

Price statistics:
count      23273.00
mean        2528.75
std        16473.90
min            4.00
25%          923.00
50%         1379.00
75%         2207.00
max      1000000.00
Name: price, dtype: float64


In [10]:
# Investigate outliers
print("Listings with price > 50,000 THB:")
print(listings[listings['price'] > 50000][['name', 'price', 'room_type', 'accommodates']].sort_values('price', ascending=False).head(10))

print(f"\nListings with price < 100 THB:")
print(listings[listings['price'] < 100][['name', 'price', 'room_type', 'accommodates']].head(10))

Listings with price > 50,000 THB:
                                                    name      price  \
979                   Modern,wifi,5m MRT&2 Shopping Mall  1000000.0   
1442                  2 Bedroom,wifi,5mMRT&Shopping Mall  1000000.0   
1946   Resort Style Luxury apartment,5min to MRT,free...  1000000.0   
3723        Modern&Luxury apartment,5min to MRT,freeWifi  1000000.0   
3276                       Ideo verse ratchaprarop condo   928572.0   
4676                                     Gemma Sukhumvit   860000.0   
15326  Loft Green Bangkok Couple or Friends Room (Max 3)   433108.0   
26664                        A comfy Home in Central BKK   321829.0   
12719                       Siri Sala Private Thai Villa   184368.0   
5162   【ExLN:9min to BTS/MRT&Terminal21,Big Balcony,Q...   153014.0   

             room_type  accommodates  
979    Entire home/apt             4  
1442   Entire home/apt             5  
1946   Entire home/apt             7  
3723   Entire home/apt      

In [11]:
# Remove outliers — keep only realistic prices
# Lower bound: 100 THB (~$3 USD) — anything below is likely a test listing
# Upper bound: 50,000 THB (~$1,400 USD) — captures luxury without data errors
# The 1,000,000 THB listings appear to be placeholder/error prices by the same host

price_lower = 100
price_upper = 50000

listings_clean = listings[
    (listings['price'] >= price_lower) &
    (listings['price'] <= price_upper)
].copy()  # .copy() prevents SettingWithCopyWarning on future modifications

print(f"Listings before outlier removal: {len(listings)}")
print(f"Listings after outlier removal:  {len(listings_clean)}")
print(f"Removed: {len(listings) - len(listings_clean)} listings ({((len(listings) - len(listings_clean)) / len(listings) * 100):.1f}%)")
print(f"\nPrice range after cleaning: {listings_clean['price'].min()} — {listings_clean['price'].max()} THB")
print(f"New mean:   {listings_clean['price'].mean():.0f} THB")
print(f"New median: {listings_clean['price'].median():.0f} THB")

Listings before outlier removal: 28806
Listings after outlier removal:  23233
Removed: 5573 listings (19.3%)

Price range after cleaning: 122.0 — 50000.0 THB
New mean:   2141 THB
New median: 1378 THB


### Data Cleaning Decisions

| Column | Issue | Action | Reason |
|--------|-------|--------|--------|
| `license`, `calendar_updated`, `neighbourhood_group_cleansed` | 100% missing | Dropped | No analytical value |
| `neighbourhood` | 67% missing | Dropped | `neighbourhood_cleansed` is the clean version |
| 8 URL/metadata columns | Not needed for analysis | Dropped | Reduce noise |
| `price` | Stored as string with `$` and `,` | Converted to float | Required for any numeric analysis |
| `price` < 100 THB | Likely test/error listings | Removed | Not realistic rental prices |
| `price` > 50,000 THB | Data entry errors or placeholders | Removed | Distort distributions and averages |

**Working dataset: 23,233 listings × 69 columns**  
**Price range: 122 — 50,000 THB**  
**Note:** ~35% of listings have no review scores — these are new listings, not bad data. They are kept in the dataset but excluded from review-based analysis.